# Corrida final — Predicción del rendimiento académico por asignatura

Este notebook ejecuta la corrida completa en Google Colab:

1. **Datos**: carga, limpieza y selección de asignaturas.
2. **Entrenamiento**: búsqueda de hiperparámetros con Optuna dentro de una validación cruzada de 5 folds y ajuste del modelo final por asignatura (RF, XGBoost y CatBoost). Es **reanudable**: si Colab se desconecta, vuelve a ejecutar las celdas y continúa donde quedó.
3. **Línea base**: regresión lineal en las 8 asignaturas de referencia.
4. **Inferencia** en el periodo reservado 2025-30.
5. **Evaluación**: tablas del documento.
6. **Estadística**: Friedman, bootstrap y cambio de distribución de notas.
7. **Tutorías**: propuesta de apoyos y su validación.
8. **Tiempo de SHAP** en una muestra pequeña.

Todos los resultados quedan como CSV en `<carpeta_base>/resultados_v3` y los modelos en `<carpeta_base>/modelos_v3`. La correspondencia de cada CSV con el documento está en el `README.md`.

## 1. Conectar Google Drive e instalar dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q optuna catboost shap xgboost

## 2. Configuración

- `CARPETA_BASE`: carpeta de Drive que contiene el parquet del historial, el Excel `Repitencia_asignaturas_punto7.xlsx` y la carpeta `codigo_corrida_final`.
- El resto de los parámetros se pueden ajustar aquí sin tocar el código.

In [ ]:
import sys
from pathlib import Path

CARPETA_BASE = Path('/content/drive/MyDrive/ProyectoGrado')   # <-- ajustar
sys.path.insert(0, str(CARPETA_BASE / 'codigo_corrida_final'))

from rendimiento import Configuracion, Corrida, configurar_registro
configurar_registro()

cfg = Configuracion(
    carpeta_base=CARPETA_BASE,
    optuna_ensayos=15,            # ensayos por modelo (ver calibración abajo)
    optuna_tiempo_max_seg=90,     # tope de la búsqueda por modelo (s); el ensayo en curso siempre termina
    metrica_optuna_clasificacion='pr_auc',
    usar_gpu=False,
    # Si CatBoost resulta muy lento en la calibración, activar:
    # catboost_parametros_fijos={'one_hot_max_size': 255},
)

## 3. (Opcional, recomendado) Calibración de tiempos

Corre todo el flujo con 3 asignaturas en una carpeta aparte (`calibracion`) y muestra cuánto tarda cada etapa. Con eso se ajustan `optuna_ensayos` y `optuna_tiempo_max_seg` para que la corrida completa quepa en el tiempo disponible.

In [ ]:
from dataclasses import replace
cfg_prueba = replace(cfg, carpeta_modelos='calibracion/modelos', carpeta_resultados='calibracion/resultados',
                     asignaturas_a_usar=['IST2088', 'MAT1101', 'FIS1033'], tutorias_top_n=3,
                     bootstrap_repeticiones=100, ks_permutaciones=199)
prueba = Corrida(cfg_prueba)
prueba.ejecutar_todo()

from rendimiento.utilidades import leer_csv
tiempos = leer_csv(cfg_prueba.ruta_resultados / 'tiempos_ejecucion.csv')
tiempos.groupby(['etapa', 'algoritmo'], dropna=False)['segundos'].agg(['sum', 'mean', 'count']).round(1)

## 4. Corrida completa (por etapas)

In [ ]:
corrida = Corrida(cfg)
corrida.etapa_datos()

In [ ]:
# La etapa más larga. Si Colab se desconecta: volver a ejecutar la celda de configuración,
# la de etapa_datos y esta; las asignaturas ya terminadas se omiten.
corrida.etapa_entrenamiento()

In [ ]:
corrida.etapa_linea_base()
corrida.etapa_inferencia()
corrida.etapa_evaluacion()

In [ ]:
corrida.etapa_estadistica()
corrida.etapa_tutorias()
corrida.etapa_tiempo_shap()

## 5. Revisión rápida de resultados

In [ ]:
from rendimiento.utilidades import leer_csv
r = cfg.ruta_resultados
display(leer_csv(r / 'tabla10_global.csv'))
display(leer_csv(r / 'punto2_anexoC_no_aprobacion.csv'))
display(leer_csv(r / 'tutorias_validacion.csv'))
display(leer_csv(r / 'tutorias_control_sensatez.csv'))

## 6. (Rutina adicional) Inferencia 2025-30 con interpretación SHAP

Usa los modelos ya entrenados (no reentrena). Genera `predicciones_2025_30_con_shap.csv` con las mismas
columnas de `predicciones_2025_30.csv` más, por algoritmo:
`interpretacion_general_<ALG>`, `interpretacion_registro_<ALG>` y `shap_valor_base_<ALG>`
(formato de las corridas anteriores). Antes de ejecutarla: celdas de montar Drive, instalar,
configuración y `corrida.etapa_datos()`.

- `algoritmos`: por defecto CatBoost y XGBoost, como antes. Agregar `'RF'` es posible pero lento.
- `exportar_valores=True` guarda además `valores_shap_2025_30.csv` (valor SHAP numérico por variable y registro).

In [ ]:
corrida = Corrida(cfg)
corrida.etapa_datos()
predicciones_shap = corrida.etapa_inferencia_con_shap(algoritmos=('CAT', 'XGB'), top_n=10, exportar_valores=True)
predicciones_shap[['Cod materia curso', 'nota_real', 'nota_estimada_CAT', 'interpretacion_registro_CAT']].head()